# 07 · Grouped folds, subgroup intervals and last-assessment type

Computes:

- D1 · students who hold more than one enrolment (§4.2)
- D2 · last-assessment type, and whether it varies within a module-presentation (§4.1, §4.4)
- D3 · refits with cross-fitting folds grouped by student, five seeds (§4.6, §5.8)
- D4 · 100 student-level bootstrap refits with grouped folds, saving subgroup means (Table S1, band test)
- D5 · Table S1, the course × band table, the band test and the last-type strata
- D6 · small figures printed for §5.3 and §5.8
- D7 · the outcome under fixed tie rules, and adjusted for last-assessment type (§4.1, §5.8)

Run notebooks 01–05 first. D2 also needs the raw OULAD files in `raw/`. D4 takes about 45 minutes and resumes from its checkpoint if the session restarts.

In [3]:
# D0  Setup
import os, json, time, gc
from pathlib import Path
import numpy as np, pandas as pd, joblib
from econml.dml import CausalForestDML, LinearDML
from sklearn.ensemble import HistGradientBoostingRegressor

root = Path.cwd()
for p in [root, *root.parents]:
    if (p / 'raw').is_dir():
        root = p
        break
BASE = str(root)
CKPT = f'{BASE}/subgroup_checkpoints'
os.makedirs(CKPT, exist_ok=True)

SEED = 42
CANONICAL_ATE = 0.004786482048864249
CONF = ['region', 'imd_band', 'highest_education', 'age_band', 'gender', 'disability',
        'code_presentation', 'num_of_prev_attempts', 'studied_credits', 'code_module']
CAT = [c for c in CONF if c not in ('num_of_prev_attempts', 'studied_credits')]
KEY = ['id_student', 'code_module', 'code_presentation']

full = pd.read_csv(f'{BASE}/oulad_full_corpus.csv')
full['imd_band'] = full['imd_band'].fillna('Missing')
X = pd.get_dummies(full[CONF], columns=CAT, drop_first=True).astype(float)
Xm = X.to_numpy()
T = full['oucontent_clicks'].to_numpy(dtype=float)
Y = full['performance_gain'].to_numpy(dtype=float)
G = full['id_student'].to_numpy()
assert X.shape == (17529, 40), X.shape

cf = joblib.load(f'{BASE}/cf_canonical_model.pkl')
assert abs(float(cf.ate(X)) - CANONICAL_ATE) < 1e-6
full['cate'] = cf.effect(X)

def make_cf(seed=SEED):
    return CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=seed),
                           model_t=HistGradientBoostingRegressor(random_state=seed),
                           discrete_treatment=False, honest=True, min_samples_leaf=50,
                           n_estimators=500, cv=5, random_state=seed)
print(f'canonical ATE {CANONICAL_ATE}   n = {len(full):,}')

canonical ATE 0.004786482048864249   n = 17,529


### D1 · Students with more than one enrolment (§4.2)

In [4]:
vc = full['id_student'].value_counts()
multi = vc[vc > 1]
print(f'enrolments {len(full):,}   distinct students {full.id_student.nunique():,}')
print(f'students with more than one enrolment {len(multi):,}, holding {int(multi.sum()):,} enrolments; max per student {int(vc.max())}')
print(f'enrolments repeating the same course {int(full.duplicated(["id_student", "code_module"]).sum()):,}')

enrolments 17,529   distinct students 16,449
students with more than one enrolment 1,071, holding 2,151 enrolments; max per student 3
enrolments repeating the same course 77


### D2 · Outcome rebuild and last-assessment type (§4.1, §4.4)
Rebuilds the outcome from the raw files: assessments that are not banked and carry weight above 0, ordered by due date, with an exam that has no recorded due date counted last. The outcome is the last recorded score minus the first. An enrolment qualifies with two or more such submissions, even when only one of them has a recorded score. Where two assessments share the first or last due date, notebook 01 left their order to the sort routine; `01_corpus/outcome_tie_choices.csv` records which one it took, so the rebuild reproduces the corpus exactly.

In [5]:
sa = pd.read_csv(f'{BASE}/raw/studentAssessment.csv', na_values=['?', ''])
asm = pd.read_csv(f'{BASE}/raw/assessments.csv', na_values=['?', ''])
TIES = pd.read_csv(f'{BASE}/01_corpus/outcome_tie_choices.csv')

def candidates(sa, asm):
    m = sa.merge(asm[['id_assessment', 'code_module', 'code_presentation', 'assessment_type', 'date', 'weight']], on='id_assessment')
    allv = m[(m['is_banked'] == 0) & (m['weight'] > 0)]
    n_sub = allv.groupby(KEY).size().rename('n').reset_index()
    v = allv.dropna(subset=['score']).copy()
    v['d'] = v['date'].fillna(np.inf)
    g = v.groupby(KEY)['d']
    v['is_first'] = v['d'].eq(g.transform('min'))
    v['is_last'] = v['d'].eq(g.transform('max'))
    return v.merge(n_sub, on=KEY)

def rebuild(v, choices=None, prefer=None):
    # choices: recorded corpus picks; prefer: 'TMA' or 'CMA' takes that type at a same-day tie
    out = []
    for end in ['first', 'last']:
        c = v[v[f'is_{end}']]
        if choices is not None:
            k = c.merge(choices[choices['end'] == end][KEY + ['id_assessment']], on=KEY, how='left', suffixes=('', '_ch'))
            c = k[(k['id_assessment_ch'].isna() | (k['id_assessment'] == k['id_assessment_ch'])).values]
        if prefer is not None:
            c = c.assign(_p=(c['assessment_type'] != prefer).astype(int)).sort_values(['_p', 'id_assessment']).drop(columns='_p')
        else:
            c = c.sort_values('id_assessment')
        c = c.drop_duplicates(KEY)
        out.append(c.set_index(KEY)[['score', 'assessment_type']].add_prefix(end + '_'))
    r = out[0].join(out[1])
    r['n'] = v.groupby(KEY)['n'].first()
    return r[r['n'] >= 2]

V = candidates(sa, asm)
rb = full.set_index(KEY)[['first_score', 'last_score', 'n_assessments']].join(rebuild(V, TIES).add_prefix('rb_'))
exact = ((rb.first_score == rb.rb_first_score) & (rb.last_score == rb.rb_last_score) & (rb.n_assessments == rb.rb_n)).mean()
print(f'rebuild reproduces first score, last score and count for {exact:.2%} of enrolments')
assert exact == 1.0
one = V.groupby(KEY).size().reindex(rb.index)
print(f'enrolments with only one recorded score (change = 0): {int((one == 1).sum())}')
t_last = TIES[TIES['end'] == 'last']
print(f'same-day ties at the last date: {len(t_last):,} enrolments; corpus took {t_last.assessment_type.value_counts().to_dict()}')
print(t_last.groupby(['code_module', 'code_presentation']).size().to_string())
full['last_type'] = rb['rb_last_assessment_type'].values
print(full['last_type'].value_counts().to_string())
per_mp = full.groupby(['code_module', 'code_presentation'])['last_type'].nunique()
print(f'module-presentations with more than one last-assessment type: {(per_mp > 1).sum()} of {len(per_mp)}')
print(full.groupby('last_type')['performance_gain_raw'].agg(['count', 'mean']).round(2).to_string())

rebuild reproduces first score, last score and count for 100.00% of enrolments
enrolments with only one recorded score (change = 0): 6
same-day ties at the last date: 2,220 enrolments; corpus took {'CMA': 1472, 'TMA': 748}
code_module  code_presentation
BBB          2013B                 469
             2013J                1054
             2014B                 697
last_type
TMA     10437
Exam     4818
CMA      2274
module-presentations with more than one last-assessment type: 9 of 19
           count   mean
last_type              
CMA         2274   8.30
Exam        4818 -11.26
TMA        10437  -6.78


### D3 · Cross-fitting folds grouped by student (§4.6, §5.8)
Refits the forest at the five seeds of the stability check, and linear DML at seed 42, with `groups` = student ID so that no student's enrolments sit on both sides of a fold boundary.

In [6]:
SEEDS = [42, 1, 7, 123, 2024]
rows = []
for s in SEEDS:
    c = make_cf(s)
    c.fit(Y, T, X=Xm, groups=G)
    lo, hi = [float(np.ravel(v)[0]) for v in c.ate_interval(Xm, alpha=0.05)]
    rows.append({'seed': s, 'ate': float(c.ate(Xm)), 'ci_low': lo, 'ci_high': hi})
    if s == SEED:
        full['cate_grouped'] = c.effect(Xm)
    print(f'seed {s:>4}: grouped ATE {rows[-1]["ate"]:.6f}   [{lo:.6f}, {hi:.6f}]', flush=True)
    del c; gc.collect()
grp = pd.DataFrame(rows)
ungrouped = pd.read_csv(f'{BASE}/multiseed_replication.csv')
print(ungrouped.to_string(index=False))
ld = LinearDML(model_y=HistGradientBoostingRegressor(random_state=SEED),
               model_t=HistGradientBoostingRegressor(random_state=SEED),
               discrete_treatment=False, cv=5, random_state=SEED)
ld.fit(Y, T, X=Xm, groups=G)
ld_lo, ld_hi = [float(np.ravel(v)[0]) for v in ld.ate_interval(Xm, alpha=0.05)]
print(f'linear DML grouped (seed 42): {float(ld.ate(Xm)):.6f}   [{ld_lo:.6f}, {ld_hi:.6f}]')
g42 = grp.set_index('seed').loc[42]
print(f'\ngrouped seed-42 ATE {g42.ate:.6f} [{g42.ci_low:.6f}, {g42.ci_high:.6f}]   canonical {CANONICAL_ATE:.6f}')
print(f'shift at seed 42 {g42.ate - CANONICAL_ATE:+.6f}')
print(f'grouped five-seed mean {grp.ate.mean():.6f} (SD {grp.ate.std(ddof=1):.6f})   ungrouped five-seed mean 0.004788 (SD 0.000057)')
print(f'shift of five-seed means {grp.ate.mean() - 0.004788:+.6f}   larger than 0.000057: {abs(grp.ate.mean() - 0.004788) > 0.000057}')
grp.to_csv(f'{BASE}/grouped_refit.csv', index=False)
pd.DataFrame([{'model': 'LinearDML grouped', 'ate': float(ld.ate(Xm)), 'ci_low': ld_lo, 'ci_high': ld_hi}]).to_csv(f'{BASE}/grouped_ldml.csv', index=False)

seed   42: grouped ATE 0.004750   [0.002554, 0.006946]
seed    1: grouped ATE 0.004840   [0.002612, 0.007068]
seed    7: grouped ATE 0.004757   [0.002391, 0.007122]
seed  123: grouped ATE 0.004817   [0.002616, 0.007019]
seed 2024: grouped ATE 0.004882   [0.002725, 0.007039]
 seed   CF_ATE  CF_width  LDML_ATE  LDML_width
   42 0.004786  0.004421  0.003706    0.001982
    1 0.004857  0.004475  0.003697    0.001970
    7 0.004793  0.004582  0.003751    0.001964
  123 0.004699  0.004363  0.003617    0.001970
 2024 0.004803  0.004182  0.003789    0.001950
linear DML grouped (seed 42): 0.003671   [0.002678, 0.004664]

grouped seed-42 ATE 0.004750 [0.002554, 0.006946]   canonical 0.004786
shift at seed 42 -0.000037
grouped five-seed mean 0.004809 (SD 0.000056)   ungrouped five-seed mean 0.004788 (SD 0.000057)
shift of five-seed means +0.000021   larger than 0.000057: False


### D4 · 100 student-level bootstrap refits
Each refit resamples students with replacement (all of a student's enrolments together), fits the canonical forest with folds grouped by student, predicts for the 17,529 original enrolments, and saves the subgroup means. Resumes from `subgroup_checkpoints/` if interrupted.

In [7]:
EDU = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent', 'HE Qualification', 'Post Graduate Qualification']
def imd_q(b):
    if b == 'Missing':
        return 'Missing'
    lo = int(str(b).split('-')[0])
    return f'Q{lo // 20 + 1} ({lo // 20 * 20}–{lo // 20 * 20 + 20}%)'
full['imd_q'] = full['imd_band'].map(imd_q)
res = full['code_module'] != 'CCC'
GROUPS = {'all': np.ones(len(full), bool)}
for e in EDU:
    GROUPS[f'edu|{e}'] = (res & (full['highest_education'] == e)).to_numpy()
for q in sorted(full['imd_q'].unique()):
    GROUPS[f'imd|{q}'] = (full['imd_q'] == q).to_numpy()
for g_ in ['F', 'M']:
    GROUPS[f'gender|{g_}'] = (full['gender'] == g_).to_numpy()
for t in sorted(full['last_type'].dropna().unique()):
    GROUPS[f'last|{t}'] = (full['last_type'] == t).to_numpy()

N_BOOT = 100
PROG = f'{CKPT}/D4_progress_v2.json'
done = json.load(open(PROG))['rows'] if os.path.exists(PROG) else []
students = full['id_student'].unique()
rows_by_student = full.groupby('id_student').indices
t0 = time.time()
for i in range(len(done), N_BOOT):
    rng = np.random.default_rng(SEED + i)
    pick = rng.choice(students, size=len(students), replace=True)
    idx = np.concatenate([rows_by_student[s] for s in pick])
    gid = np.repeat(np.arange(len(pick)), [len(rows_by_student[s]) for s in pick])
    c = make_cf(SEED)
    c.fit(Y[idx], T[idx], X=Xm[idx], groups=gid)
    eff = c.effect(Xm)
    done.append({k: float(eff[v].mean()) for k, v in GROUPS.items()})
    del c; gc.collect()
    print(f'[{i + 1:>3}/{N_BOOT}] ATE {done[-1]["all"]:.6f}   elapsed {(time.time() - t0) / 60:5.1f} min', flush=True)
    if (i + 1) % 5 == 0 or i + 1 == N_BOOT:
        json.dump({'rows': done}, open(PROG, 'w'))
boot = pd.DataFrame(done)
assert len(boot) == N_BOOT
boot.to_csv(f'{BASE}/subgroup_bootstrap.csv', index=False)
print(f'bootstrap ATE: mean {boot["all"].mean():.6f}   SD {boot["all"].std(ddof=1):.6f}   95% [{boot["all"].quantile(.025):.6f}, {boot["all"].quantile(.975):.6f}]')

[  1/100] ATE 0.004254   elapsed   0.5 min
[  2/100] ATE 0.004501   elapsed   0.9 min
[  3/100] ATE 0.004901   elapsed   1.4 min
[  4/100] ATE 0.004787   elapsed   1.9 min
[  5/100] ATE 0.004659   elapsed   2.3 min
[  6/100] ATE 0.005147   elapsed   2.7 min
[  7/100] ATE 0.004351   elapsed   3.2 min
[  8/100] ATE 0.004781   elapsed   3.6 min
[  9/100] ATE 0.004953   elapsed   4.1 min
[ 10/100] ATE 0.004466   elapsed   4.5 min
[ 11/100] ATE 0.004361   elapsed   5.0 min
[ 12/100] ATE 0.004355   elapsed   5.5 min
[ 13/100] ATE 0.005025   elapsed   5.9 min
[ 14/100] ATE 0.004887   elapsed   6.4 min
[ 15/100] ATE 0.004578   elapsed   6.8 min
[ 16/100] ATE 0.004947   elapsed   7.3 min
[ 17/100] ATE 0.005229   elapsed   7.7 min
[ 18/100] ATE 0.004775   elapsed   8.3 min
[ 19/100] ATE 0.004652   elapsed   8.8 min
[ 20/100] ATE 0.004339   elapsed   9.2 min
[ 21/100] ATE 0.004591   elapsed   9.7 min
[ 22/100] ATE 0.004445   elapsed  10.2 min
[ 23/100] ATE 0.004444   elapsed  10.6 min
[ 24/100] A

### D5 · Table S1, course × band table, band test, last-type strata
Point estimates use the canonical (seed-42) estimates, the same ones as Table 4. Intervals are the 2.5th and 97.5th percentiles of the 100 refits. Required n uses the Table 6 formula: n × (half-width / |mean|)².

In [8]:
tab = []
for k, mask in GROUPS.items():
    if k == 'all':
        continue
    dim, lev = k.split('|')
    v = full.loc[mask, 'cate']
    lo, hi = boot[k].quantile(0.025), boot[k].quantile(0.975)
    hw = (hi - lo) / 2
    tab.append({'dimension': dim, 'level': lev, 'n': int(mask.sum()), 'mean': v.mean(),
                'ci_low': lo, 'ci_high': hi, 'excludes_zero': bool(lo > 0 or hi < 0),
                'required_n': int(np.ceil(mask.sum() * (hw / abs(v.mean())) ** 2)),
                'pct_negative': 100 * (v < 0).mean()})
S1 = pd.DataFrame(tab)
pd.set_option('display.width', 200)
print(S1.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
S1.to_csv(f'{BASE}/table_s1.csv', index=False)

d = boot['edu|Lower Than A Level'] - boot['edu|HE Qualification']
point = S1.set_index('level').loc['Lower Than A Level', 'mean'] - S1.set_index('level').loc['HE Qualification', 'mean']
print(f'\nband test, below A level minus HE: point {point:.6f}   95% [{d.quantile(.025):.6f}, {d.quantile(.975):.6f}]   '
      f'refits with difference <= 0: {(d <= 0).sum()} of {len(d)}   excludes zero: {bool(d.quantile(.025) > 0 or d.quantile(.975) < 0)}')

cb = full[res].pivot_table(index='code_module', columns='highest_education', values='cate', aggfunc='mean').reindex(columns=EDU)
cn = full[res].pivot_table(index='code_module', columns='highest_education', values='cate', aggfunc='size').reindex(columns=EDU)
print('\ncourse × band means'); print(cb.to_string(float_format=lambda x: f'{x:.6f}'))
print('\ncourse × band n'); print(cn.to_string())
held = [set(r.dropna().nsmallest(2).index) == {'HE Qualification', 'Post Graduate Qualification'} for _, r in cb.iterrows()]
print(f'courses where HE and postgraduate are the two lowest bands: {sum(held)} of {len(held)}')
lta_top = [r.dropna().idxmax() for _, r in cb.iterrows()]
print('highest band per course:', dict(zip(cb.index, lta_top)))
cb.to_csv(f'{BASE}/course_band_means.csv'); cn.to_csv(f'{BASE}/course_band_n.csv')

dimension                       level     n     mean   ci_low  ci_high  excludes_zero  required_n  pct_negative
      edu             No Formal quals   105 0.005430 0.004681 0.006239           True           3      0.000000
      edu          Lower Than A Level  5409 0.005759 0.005015 0.006314           True          69      0.000000
      edu       A Level or Equivalent  7230 0.005443 0.004645 0.006152           True         139      0.000000
      edu            HE Qualification  2529 0.004702 0.003916 0.005419           True          65      0.000000
      edu Post Graduate Qualification   142 0.004680 0.003863 0.005282           True           4      0.000000
      imd                     Missing   830 0.004041 0.003243 0.004687           True          27      6.626506
      imd                  Q1 (0–20%)  3068 0.005217 0.004569 0.005926           True          52      2.281617
      imd                 Q2 (20–40%)  3624 0.004843 0.004176 0.005404           True          59      4

### D6 · Figures printed for §5.3, §5.8 and §6.6

In [9]:
print(f'n_F = {(full.gender == "F").sum():,}   n_M = {(full.gender == "M").sum():,}')
nul = pd.read_csv(f'{BASE}/calibration_null.csv')
print(f'constant-effect null: variance mean {nul.null_variance.mean():.4e}   SD {nul.null_variance.std(ddof=1):.4e}')
strip = pd.read_csv(f'{BASE}/leaf_sensitivity_strip.csv')
print(strip.to_string(index=False))
dec = full['cate'] <= full['cate'].quantile(0.10)
print(f'\nlowest-decile estimates ({int(dec.sum()):,} enrolments) by course:')
print(full.loc[dec, 'code_module'].value_counts().to_string())
print('by prior education:'); print(full.loc[dec, 'highest_education'].value_counts().to_string())
print('by deprivation quintile:'); print(full.loc[dec, 'imd_q'].value_counts().to_string())
print('by gender:'); print(full.loc[dec, 'gender'].value_counts().to_string())
print(f'grouped vs canonical per-enrolment estimates: correlation {np.corrcoef(full.cate, full.cate_grouped)[0, 1]:.4f}')

n_F = 7,074   n_M = 10,455
constant-effect null: variance mean 4.3675e-07   SD 9.9709e-08
 min_samples_leaf      ATE    CI_low  CI_high  CI_width  CATE_sd  pct_negative excludes_zero
                5 0.004731 -0.001127 0.010588  0.011715 0.003079      0.078498            no
               15 0.004732  0.001358 0.008106  0.006748 0.002432      0.064522           yes
               25 0.004758  0.002014 0.007501  0.005487 0.002261      0.060414           yes
               50 0.004786  0.002576 0.006997  0.004421 0.002129      0.045011           yes
              100 0.004810  0.002978 0.006642  0.003664 0.002058      0.026756           yes

lowest-decile estimates (1,753 enrolments) by course:
code_module
CCC    1753
by prior education:
highest_education
A Level or Equivalent          830
HE Qualification               483
Lower Than A Level             349
Post Graduate Qualification     80
No Formal quals                 11
by deprivation quintile:
imd_q
Q5 (80–100%)    384
Q3 (40–60

### D7 · Outcome under fixed tie rules; estimate adjusted for last-assessment type (§4.1, §5.8)
Refits the canonical forest at the five seeds with (a) the tutor-marked score always taken at a same-day tie, (b) the computer-marked score always taken, and (c) the corpus outcome with last-assessment type added to the confounders. Winsorisation keeps the corpus bounds.

In [10]:
lo_w, hi_w = float(full['performance_gain'].min()), float(full['performance_gain'].max())
print(f'winsorisation bounds kept from the corpus: [{lo_w:.2f}, {hi_w:.2f}]')
variants = {}
for rule in ['TMA', 'CMA']:
    r = full.set_index(KEY)[[]].join(rebuild(V, None, prefer=rule))
    y = (r['last_score'] - r['first_score']).clip(lo_w, hi_w).to_numpy(dtype=float)
    print(f'rule {rule}: enrolments whose outcome changes {int((y != Y).sum()):,}')
    variants[f'tie rule {rule}'] = (y, Xm)
Xl = np.column_stack([Xm, pd.get_dummies(full['last_type'], drop_first=True).astype(float).to_numpy()])
variants['adjusted for last type'] = (Y, Xl)
rows7 = []
for name, (y, xx) in variants.items():
    for s in [42, 1, 7, 123, 2024]:
        c = make_cf(s); c.fit(y, T, X=xx)
        lo, hi = [float(np.ravel(v)[0]) for v in c.ate_interval(xx, alpha=0.05)]
        rows7.append({'variant': name, 'seed': s, 'ate': float(c.ate(xx)), 'ci_low': lo, 'ci_high': hi})
        print(f'{name:<24} seed {s:>4}: {rows7[-1]["ate"]:.6f}  [{lo:.6f}, {hi:.6f}]', flush=True)
        del c; gc.collect()
r7 = pd.DataFrame(rows7)
print(r7.groupby('variant')['ate'].agg(['mean', 'std']).to_string(float_format=lambda x: f'{x:.6f}'))
print('ungrouped corpus five-seed mean 0.004788 (SD 0.000057)')
r7.to_csv(f'{BASE}/outcome_variants.csv', index=False)

winsorisation bounds kept from the corpus: [-61.38, 40.00]
rule TMA: enrolments whose outcome changes 1,469
rule CMA: enrolments whose outcome changes 731
tie rule TMA             seed   42: 0.004788  [0.002567, 0.007010]
tie rule TMA             seed    1: 0.004774  [0.002623, 0.006924]
tie rule TMA             seed    7: 0.004777  [0.002478, 0.007077]
tie rule TMA             seed  123: 0.004680  [0.002469, 0.006891]
tie rule TMA             seed 2024: 0.004727  [0.002556, 0.006898]
tie rule CMA             seed   42: 0.004835  [0.002670, 0.007000]
tie rule CMA             seed    1: 0.004850  [0.002624, 0.007076]
tie rule CMA             seed    7: 0.004876  [0.002546, 0.007205]
tie rule CMA             seed  123: 0.004746  [0.002562, 0.006930]
tie rule CMA             seed 2024: 0.004811  [0.002704, 0.006919]
adjusted for last type   seed   42: 0.004343  [0.001762, 0.006924]
adjusted for last type   seed    1: 0.004294  [0.001387, 0.007202]
adjusted for last type   seed    7: 0.004